# Aula 4 — Sinal v1: Momentum Cross-Seccional

**Intensivão Quant AI — ImpactUFSCar**

---

Esta é a aula central do intensivão. Você vai construir o sinal que vai gerar todos os sinais de compra e venda da estratégia. Tudo que vem antes foi preparação; tudo que vem depois é refinamento e validação.

Ao final, você terá:
- Entendido por que momentum funciona através de probabilidade condicional
- Calculado o sinal 12-1 para todas as ações do IBOVESPA
- Montado a primeira carteira com equal weight

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Dados limpos da Aula 3
ret_mensais = pd.read_parquet('retornos_mensais_limpo.parquet')
tickers     = pd.read_csv('tickers_finais.csv')['ticker'].tolist()

print(f"Dataset: {ret_mensais.shape[0]} meses × {ret_mensais.shape[1]} ações")
print(f"Período: {ret_mensais.index[0].date()} a {ret_mensais.index[-1].date()}")

## 1. Por que momentum funciona — a visão de probabilidade condicional

Antes de qualquer código, precisamos entender **por que** a estratégia deveria funcionar. Sem tese, não há estratégia — só data mining.

### Probabilidade condicional

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

A probabilidade condicional responde: *dado que B aconteceu, qual a probabilidade de A?*

A aposta do momentum é:

$$P(\text{retorno futuro alto} \mid \text{retorno passado alto}) > P(\text{retorno futuro alto})$$

Ou seja: saber que uma ação performou bem no passado deve **aumentar** a probabilidade de ela performar bem no futuro — comparado com uma ação qualquer.

### Por que isso seria verdade?

Três mecanismos comportamentais explicados por Jegadeesh & Titman (1993):

1. **Underreaction:** quando uma empresa publica um bom resultado, analistas e investidores demoram a ajustar suas expectativas. O preço sobe, mas devagar — e continua subindo por meses
2. **Herding:** à medida que o ativo sobe, mais investidores notam e entram. O fluxo de capital prolonga o movimento
3. **Ancoragem:** investidores ancoram seus preços-alvo em preços históricos, subestimando o quanto a situação mudou

Vamos verificar empiricamente nos dados do IBOVESPA.

In [ ]:
# Teste empírico de probabilidade condicional
# Pergunta: ações no quartil superior do retorno passado (3 meses)
#           têm retorno futuro (1 mês) maior que as do quartil inferior?

# Retorno dos últimos 3 meses (sinal simples)
ret_passado = ret_mensais.shift(1).rolling(3).sum()

# Retorno do próximo mês (o que queremos prever)
ret_futuro = ret_mensais

# Para cada mês, classifica as ações em quartis pelo retorno passado
resultados = []
for data in ret_passado.index:
    passado = ret_passado.loc[data].dropna()
    futuro  = ret_futuro.loc[data].dropna() if data in ret_futuro.index else pd.Series()

    ativos_comuns = passado.index.intersection(futuro.index)
    if len(ativos_comuns) < 10:
        continue

    p = passado[ativos_comuns]
    f = futuro[ativos_comuns]

    quartis = pd.qcut(p, 4, labels=['Q1 (piores)', 'Q2', 'Q3', 'Q4 (melhores)'])
    for q in ['Q1 (piores)', 'Q4 (melhores)']:
        mask = quartis == q
        if mask.sum() > 0:
            resultados.append({'data': data, 'quartil': q, 'retorno_futuro': f[mask].mean()})

df_resultados = pd.DataFrame(resultados)

# Média por quartil
media_por_quartil = df_resultados.groupby('quartil')['retorno_futuro'].mean()
print("Retorno médio no mês seguinte por quartil de momentum passado (3m):")
for q, v in media_por_quartil.items():
    print(f"  {q}: {v:.2%} ao mês")
print()
print(f"Diferença Q4 − Q1: {media_por_quartil['Q4 (melhores)'] - media_por_quartil['Q1 (piores)']:.2%} ao mês")

In [ ]:
# Visualização: distribuição dos retornos futuros por quartil
fig, ax = plt.subplots(figsize=(11, 5))

for quartil, grupo in df_resultados.groupby('quartil'):
    grupo['retorno_futuro'].hist(
        ax=ax, bins=40, alpha=0.5, density=True, label=quartil
    )

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Distribuição do retorno futuro por quartil de momentum passado')
ax.set_xlabel('Retorno no mês seguinte')
ax.set_ylabel('Densidade')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Cross-seccional vs time-series momentum

Existem duas formas de usar momentum. É importante entender a diferença porque elas respondem perguntas diferentes.

### Time-series momentum

Cada ativo é comparado com **ele mesmo** no passado:
- Se PETR4 subiu mais do que o normal nos últimos 12 meses → compra PETR4
- Se PETR4 caiu → fica de fora (ou vende a descoberto)

Pergunta respondida: *"Essa ação está em tendência positiva?"*

### Cross-seccional momentum

Cada ativo é comparado com os **outros ativos no mesmo período**:
- Ranqueia todas as ações pelo retorno dos últimos 12 meses
- Compra as do topo; ignora (ou vende) as do fundo
- O sinal é **relativo**: não importa se o mercado subiu ou caiu, importa quem subiu **mais**

Pergunta respondida: *"Essa ação se destacou positivamente vs as peers?"*

**Vamos usar cross-seccional.** É mais natural para carteiras long-only, menos sensível à direção geral do mercado, e é o que o 1º lugar de 2024 usou.

## 3. Por que a janela 12-1?

Jegadeesh & Titman (1993) testaram todas as combinações de janela de formação (3 a 12 meses) e horizonte de holding (3 a 12 meses). O sweet spot empírico ficou em **formação de 12 meses, holding de 3 a 6 meses**.

**O skip de 1 mês:** se você inclui o mês mais recente no sinal, você acaba capturando um efeito oposto — **reversão de curto prazo**. Ações que subiram muito no último mês tendem a reverter no mês seguinte (por causa de bid-ask bounce e microestrutura de mercado). Ao excluir o mês mais recente, você limpa o sinal.

```
Timeline do sinal para o mês de rebalanceamento M:

M-13    M-12    M-11  ...  M-2    M-1    M (agora)
  |-------- janela do sinal (11 meses) ---------|  skip
```

O sinal = retorno acumulado de M-12 a M-2 (11 meses, pulando M-1).

In [ ]:
# Calculando o sinal 12-1
# Em cada mês M, o sinal usa retornos de M-12 até M-2 (11 meses, skip de 1)
# shift(2): o retorno mais recente na janela foi 2 meses atrás
# rolling(11): soma 11 meses consecutivos

sinal = ret_mensais.shift(2).rolling(11).sum()

print("Sinal calculado.")
print(f"Shape: {sinal.shape}")
print(f"Primeiros dados disponíveis: {sinal.dropna(how='all').index[0].date()}")
print(f"  (13 meses após o início dos retornos — esperado)")
print()
print("Sinal do último mês disponível — top 5:")
ultimo_mes = sinal.iloc[-1].dropna().sort_values(ascending=False)
print(ultimo_mes.head().rename(lambda x: x.replace('.SA','')))

In [ ]:
# Visualizando o sinal ao longo do tempo para algumas ações
acoes_exemplo = ['PETR4.SA', 'VALE3.SA', 'WEGE3.SA', 'MGLU3.SA']
acoes_plot    = [a for a in acoes_exemplo if a in sinal.columns]

fig, ax = plt.subplots(figsize=(13, 5))
for acao in acoes_plot:
    sinal[acao].plot(ax=ax, label=acao.replace('.SA', ''), alpha=0.8)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Sinal de momentum 12-1 ao longo do tempo')
ax.set_ylabel('Retorno acumulado (11 meses, t-2)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Ranking cross-seccional

Em cada mês, ranqueamos todas as ações pelo sinal. Quem tem o maior sinal recebe rank mais alto.

Usamos **rank percentual** (0 a 1): facilita comparar entre meses com número diferente de ações válidas.

- Rank = 1.0 → melhor sinal do mês
- Rank = 0.0 → pior sinal do mês

In [ ]:
# Rank percentual — calculado linha a linha (cada mês é uma linha)
ranking = sinal.rank(axis=1, ascending=True, pct=True)

print("Ranking percentual — último mês disponível (top 5 e bottom 5):")
ult_ranking = ranking.iloc[-1].dropna().sort_values(ascending=False)

print("\nTop 5 (maiores ranks):")
print(ult_ranking.head().rename(lambda x: x.replace('.SA','')).round(3))

print("\nBottom 5 (menores ranks):")
print(ult_ranking.tail().rename(lambda x: x.replace('.SA','')).round(3))

## 5. Long-only vs long-short

### Long-short (versão acadêmica)
- Compra o top quintil (os 20% com maior sinal)
- Vende a descoberto o bottom quintil (os 20% com menor sinal)
- O lucro vem da diferença entre os dois grupos
- Mais "puro" — isola o efeito de momentum do risco de mercado

### Long-only (versão prática — o que usamos)
- Compra apenas o top N
- Quando o mercado cai, a carteira cai junto (não há hedge do lado short)
- Mais simples de implementar, mais realista para um fundo iniciante

**Para o desafio, usamos long-only.** A banca não penaliza por isso — penaliza por backtest desonesto.

## 6. Montando a carteira — Equal Weight

Selecionamos as **top N ações** em cada mês e damos peso igual a cada uma: cada ação recebe $1/N$ do portfólio.

Vantagens do equal weight:
- Simples e transparente
- Não depende de estimativas de covariância (que são ruidosas)
- Diversificação máxima dentro do universo selecionado

Na Aula 6 vamos comparar com signal-weighted e outras abordagens.

In [ ]:
def calcular_pesos_equal_weight(ranking_row, N=10):
    """Seleciona top N ações e dá peso igual a cada uma."""
    validos = ranking_row.dropna()
    if len(validos) < N:
        return pd.Series(0.0, index=ranking_row.index)

    threshold = validos.nlargest(N).min()  # menor rank entre os top N
    selecionados = validos[validos >= threshold].index[:N]  # garante exatamente N

    pesos = pd.Series(0.0, index=ranking_row.index)
    pesos[selecionados] = 1.0 / N
    return pesos

N_ATIVOS = 10
pesos = ranking.apply(calcular_pesos_equal_weight, axis=1, N=N_ATIVOS)

# Verificação: pesos somam 1 (ou 0 nos meses sem sinal)
soma_pesos = pesos.sum(axis=1)
meses_com_carteira = (soma_pesos > 0).sum()
print(f"Meses com carteira formada: {meses_com_carteira} de {len(pesos)}")
print(f"Soma dos pesos (deve ser 1.0): {soma_pesos[soma_pesos > 0].mean():.4f}")

In [ ]:
# Quais ações aparecem mais frequentemente no top 10?
frequencia = (pesos > 0).sum().sort_values(ascending=False)
total_meses = meses_com_carteira

fig, ax = plt.subplots(figsize=(13, 4))
(frequencia.head(20) / total_meses * 100).plot(
    kind='bar', ax=ax, color='steelblue', width=0.8
)
ax.set_title(f'Frequência no top {N_ATIVOS} — % dos meses em que a ação foi selecionada')
ax.set_ylabel('% dos meses')
ax.set_xticklabels(
    [t.replace('.SA', '') for t in frequencia.head(20).index], rotation=45
)
plt.tight_layout()
plt.show()

In [ ]:
# Composição da carteira no último mês
ult_pesos = pesos.iloc[-1]
carteira_atual = ult_pesos[ult_pesos > 0].rename(lambda x: x.replace('.SA',''))

print(f"Carteira momentum — {pesos.index[-1].strftime('%b/%Y')}:")
for acao, peso in carteira_atual.items():
    print(f"  {acao:<12} {peso:.1%}")

In [ ]:
# Turnover mensal — quantas ações trocam a cada mês?
# Métrica importante: alto turnover = altos custos de transação (veremos na Aula 8)

carteira_binaria = (pesos > 0).astype(int)
entradas = (carteira_binaria.diff() > 0).sum(axis=1)   # ações que entraram
saidas   = (carteira_binaria.diff() < 0).sum(axis=1)   # ações que saíram
turnover = ((entradas + saidas) / 2) / N_ATIVOS          # % da carteira renovada

print(f"Turnover médio mensal: {turnover.mean():.1%} da carteira")
print(f"  → {turnover.mean() * N_ATIVOS:.1f} ações trocam por mês em média (de {N_ATIVOS})")

fig, ax = plt.subplots(figsize=(13, 3))
turnover.plot(ax=ax, color='steelblue', alpha=0.8)
ax.axhline(turnover.mean(), color='red', linestyle='--',
           label=f'média: {turnover.mean():.1%}')
ax.set_title('Turnover mensal da carteira')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

## 7. Retorno da carteira

Com os pesos calculados, podemos calcular o retorno mensal da carteira:

$$r_{\text{carteira},t} = \sum_{i} w_{i,t} \cdot r_{i,t}$$

Este é o produto vetorial entre pesos e retornos — **álgebra linear que você vai expandir na Aula 6**. Por ora, calculamos e plotamos o resultado.

In [ ]:
# Retorno mensal da carteira = soma ponderada dos retornos
# pesos contém os pesos decididos NO INÍCIO do mês
# ret_mensais contém os retornos REALIZADOS no mês
# Alinhamento: os pesos de M são aplicados ao retorno de M

ret_carteira = (pesos * ret_mensais).sum(axis=1)

# Remove meses sem carteira formada
ret_carteira = ret_carteira[pesos.sum(axis=1) > 0]

# Retorno acumulado
acum_carteira = ret_carteira.cumsum()

# Benchmark: IBOVESPA
import yfinance as yf
ibov_raw = yf.download('^BVSP', start='2010-01-01', end='2024-12-31',
                        auto_adjust=True, progress=False)['Close'].squeeze()
ibov_ret  = np.log(ibov_raw / ibov_raw.shift(1))
ibov_mens = ibov_ret.resample('ME').sum()
ibov_acum = ibov_mens.reindex(acum_carteira.index).cumsum()

fig, ax = plt.subplots(figsize=(13, 5))
acum_carteira.plot(ax=ax, label=f'Momentum top {N_ATIVOS} (equal weight)', linewidth=2)
ibov_acum.plot(ax=ax, label='IBOVESPA', linewidth=1.5, linestyle='--', color='gray')
ax.set_title('Retorno acumulado — Momentum v1 vs IBOVESPA')
ax.set_ylabel('Retorno log acumulado')
ax.legend()
plt.tight_layout()
plt.show()

print("Preview das métricas (versão completa na Aula 5):")
vol_mens = ret_carteira.std()
sharpe_aprox = (ret_carteira.mean() / vol_mens) * np.sqrt(12)
print(f"  Retorno médio mensal: {ret_carteira.mean():.2%}")
print(f"  Sharpe anualizado (aprox): {sharpe_aprox:.2f}")

## 8. Salvando o sinal v1

In [ ]:
sinal.to_parquet('sinal_v1.parquet')
pesos.to_parquet('pesos_v1.parquet')
ret_carteira.to_frame('ret_momentum').to_parquet('retorno_carteira_v1.parquet')

print("Arquivos salvos:")
print("  sinal_v1.parquet           — sinal 12-1 para todas as ações")
print("  pesos_v1.parquet           — pesos mensais (equal weight top 10)")
print("  retorno_carteira_v1.parquet — retorno mensal da carteira")

## Resumo da aula

| Conceito | Implementação |
|---|---|
| Probabilidade condicional | Q4 de momentum tem retorno futuro maior que Q1 — verificado empiricamente |
| Cross-seccional | Rankear ações entre si a cada mês |
| Janela 12-1 | `ret_mensais.shift(2).rolling(11).sum()` |
| Equal weight | Top 10, cada um com 10% da carteira |
| Turnover | ~30-50% ao mês — alto, vai custar na Aula 8 |

---

## Para replicar em casa

1. Rode o notebook do início ao fim
2. Experimente mudar `N_ATIVOS` para 5, 15, 20 — como muda o retorno acumulado?
3. Experimente mudar a janela do sinal: substitua `shift(2).rolling(11)` por `shift(2).rolling(5)` (6-1) ou `shift(2).rolling(8)` (9-1) — qual funciona melhor?

> **Atenção:** testar muitas combinações e ficar com a melhor é overfitting. Na Aula 8 você vai aprender a corrigir isso. Por ora, explore livremente — mas anote que está explorando.

---

*ImpactUFSCar — Diretoria de Quant*